# Rendu Blender — réseau de neurones (B01-S03)
1. **Exécution → Modifier le type d'exécution → GPU (T4)** puis **Exécuter tout**.
2. Quand la cellule « script » le demande : **verse `neural_net.py`** (icône dossier ←, bouton upload). Le fichier est dans le dépôt, dossier `blender/`.
3. Le mp4 final se télécharge tout seul à la fin. Durée estimée : ~15 min (setup + 144 frames EEVEE).

In [ ]:
!nvidia-smi -L

In [ ]:
# Blender 5.2.1 officiel (même version que le script) + X virtuel pour EEVEE
!apt-get install -y -qq xvfb > /dev/null 2>&1
!mkdir -p /tmp/blender && curl -L --retry 3 -o /tmp/blender.tar.xz https://download.blender.org/release/Blender5.2/blender-5.2.1-linux-x64.tar.xz
!tar -xJf /tmp/blender.tar.xz -C /tmp/blender --strip-components=1 && /tmp/blender/blender --version | head -1
!ls -lh /content/neural_net.py

In [ ]:
# Rendu : 144 frames, 1080p, Cycles GPU (OPTIX) sur T4
!mkdir -p /content/out && /tmp/blender/blender -b --python /content/neural_net.py -- --frames 144 --res 100 --samples 24 --engine cycles --device OPTIX --out /content/out/neural_net

In [ ]:
# Même grade que le reste du film : glow + contraste + vignette + grain
!ffmpeg -y -loglevel error -framerate 24 -i /content/out/neural_net/frame-%04d.png -vf "split[a][b];[b]gblur=sigma=10[b];[a][b]blend=screen,eq=saturation=0.85:contrast=1.06,vignette=PI/4,colorbalance=bs=0.10:bm=0.04,noise=alls=8:allf=t" -c:v libx264 -preset medium -crf 18 -pix_fmt yuv420p -movflags +faststart /content/neural_net.mp4
!ls -lh /content/neural_net.mp4
from google.colab import files
files.download('/content/neural_net.mp4')

## Variante Cycles GPU (plus beau, plus lent)
Remplacer la cellule de rendu par : `... --engine cycles --device OPTIX --samples 128 ...` (~30-60 min). EEVEE suffit pour ce plan.